# Mycetoma AI — Training Pipeline

End-to-end training notebook: setup → data → SSL pretraining → multi-task finetuning → evaluation.

**Runtime**: GPU recommended (T4/A100)

## 1. Environment Setup

In [ ]:
!git clone https://github.com/yashnaiduu/MycetomaAi.git
%cd /content/MycetomaAi

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
%env PYTHONPATH=.:$PYTHONPATH

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Data Preparation

**Option A**: Download real data from Kaggle (requires Kaggle API key).

**Option B**: Generate synthetic data for pipeline testing.

In [ ]:
# Option A: Kaggle download (uncomment if you have API key)
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d supporoot/mycetoma-ai-pretraining-data --unzip -p data/

In [ ]:
# Option B: Generate synthetic test data
!python scripts/create_sample_data.py --output_dir data/finetune --images_per_class 20 --size 224

In [ ]:
# Verify data structure
import os
for root, dirs, files in os.walk("data/finetune"):
    level = root.replace("data/finetune", "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:
        for f in files[:3]:
            print(f"{indent}  {f}")
        if len(files) > 3:
            print(f"{indent}  ... ({len(files)} total)")

## 3. Data Verification

In [ ]:
from src.data.dataset import MycetomaDataset
from src.data.transforms import get_supervised_transforms

dataset, class_map = MycetomaDataset.from_directory(
    "data/finetune",
    transform=get_supervised_transforms()["train"],
    generate_masks=True,
)

print(f"Dataset size: {len(dataset)}")
print(f"Classes: {class_map}")

sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Label: {sample['label']}")
print(f"Mask shape: {sample['mask'].shape}")
print(f"Mask range: [{sample['mask'].min():.2f}, {sample['mask'].max():.2f}]")

## 4. SSL Pretraining (Optional)

Skip this if you want to go straight to finetuning.

In [ ]:
# SSL pretraining — 5 epochs demo
!python scripts/train.py \
    --stage pretrain \
    --pretrain_data_dir data/finetune \
    --epochs 5 \
    --batch_size 8 \
    --lr 1e-4 \
    --num_workers 2 \
    --precision fp16

## 5. Multi-Task Finetuning

In [ ]:
# Finetuning — 10 epochs demo
# Add --checkpoint checkpoints/ssl/last.pth to load SSL weights
!python scripts/train.py \
    --stage finetune \
    --finetune_data_dir data/finetune \
    --epochs 10 \
    --batch_size 8 \
    --lr 1e-4 \
    --num_workers 2 \
    --precision fp16

## 6. Evaluation

In [ ]:
!python scripts/evaluate.py \
    --model_path checkpoints/multitask/best_multi_task_model.pth \
    --data_dir data/finetune \
    --batch_size 8

In [ ]:
# Display results
import json

with open("results/evaluation_results.json") as f:
    results = json.load(f)

for k, v in results.items():
    if k != "Confusion_Matrix":
        print(f"{k}: {v}")

# Show confusion matrix
from IPython.display import Image, display
if os.path.exists("results/confusion_matrix.png"):
    display(Image(filename="results/confusion_matrix.png"))

## 7. Quick Inference

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from src.models.model import MycetomaAIModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MycetomaAIModel(mode="finetune", pretrained_backbone=False).to(device)

ckpt = torch.load("checkpoints/multitask/best_multi_task_model.pth", map_location=device, weights_only=False)
model.load_state_dict(ckpt["model"], strict=False)
model.eval()

sample = dataset[0]
img = sample["image"].unsqueeze(0).to(device)

with torch.no_grad():
    preds = model(img)

cls_probs = torch.softmax(preds["classification"], dim=1).cpu().numpy()[0]
seg_mask = preds["segmentation"].cpu().numpy()[0, 0]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(list(class_map.values()), cls_probs)
axes[0].set_title("Classification")
axes[0].set_ylabel("Probability")

axes[1].imshow(seg_mask, cmap="hot")
axes[1].set_title("Segmentation")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"Predicted class: {list(class_map.values())[np.argmax(cls_probs)]}")
print(f"Confidence: {cls_probs.max():.4f}")
print(f"Mask coverage: {(seg_mask > 0.5).mean():.2%}")

## 8. Full Pipeline Validation

In [ ]:
!python scripts/validate_pipeline.py